In [1]:
from huggingface_hub import login
login(new_session=False) #token needs to be provided

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token 
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id

model.eval()

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
2026-02-08 14:50:24.172803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770562224.389631      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770562224.450454      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770562224.918610      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770562224.918637      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770562224.918640      55

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [3]:
import pandas as pd

df2 = pd.read_csv("/kaggle/input/data-nmt-2-all2/data_nmt_2_all2.csv")

In [4]:
if "llama3.1-IT-8B" not in df2.columns:
    df2["llama3.1-IT-8B"] = ""

In [5]:
def clean_translation(text: str) -> str:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else ""

In [8]:
def translate(sentence):
    messages = [
        {
            "role": "user",
            "content": 
                f"Translate this sentence from English to Turkish:\n"
                f"{sentence}\n"
                f"Output only the Turkish translation, without any additional explanations.\n"
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)
    attention_mask = torch.ones_like(inputs)

    with torch.no_grad():
        output = model.generate(
            input_ids=inputs,
            attention_mask=attention_mask, 
            max_new_tokens=64,
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    return clean_translation(decoded)
    
BATCH_SIZE = 25

for i in range(0, len(df2), BATCH_SIZE):
    batch = df2.iloc[i:i + BATCH_SIZE]

    for idx, sentence in batch["source_sentence"].items():
        translation = translate(sentence)
        df2.at[idx, "llama3.1-IT-8B"] = translation

    torch.cuda.empty_cache()
    
    print(f"Processed rows {i}–{i + len(batch) - 1}")


Processed rows 0–24
Processed rows 25–49
Processed rows 50–74
Processed rows 75–99
Processed rows 100–124
Processed rows 125–149
Processed rows 150–174
Processed rows 175–191


In [9]:
df2.to_csv("data_nmt_2_all3.csv", index=False)